# Business Entity Resolution - AWS SageMaker / Jupyter Training & Inference Pipeline
**ML Challenge 2026**

This notebook runs the complete end-to-end entity resolution workflow:
1. **Environment Setup & Verification**
2. **LightGBM Model Training & Out-of-Fold Threshold Tuning**
3. **Country-Partitioned Memory-Efficient Inference on Test Data**
4. **Strict Format & Completeness Validation**
5. **Packaging into `submission.zip`**

## 1. Environment Setup & Dependency Installation

In [ ]:
import os
import sys

# Ensure working directory is project root
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())
print(f"Current working directory: {os.getcwd()}")

# Install dependencies
!pip install -r requirements.txt --quiet
print("Dependencies installed successfully.")

## 2. Verify Processed Parquet Datasets

In [ ]:
import polars as pl

data_dir = "data/processed"
files = [
    "train_source1.parquet", "train_source2.parquet", "train_source3.parquet", "train_ground_truth.parquet",
    "test_source1.parquet", "test_source2.parquet", "test_source3.parquet"
]

print("Checking dataset availability:")
for f in files:
    path = os.path.join(data_dir, f)
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / (1024 * 1024)
        print(f"  [OK] {f} ({size_mb:.1f} MB)")
    else:
        print(f"  [MISSING] {f}")

## 3. Train LightGBM Matching Model
Trains candidate pairs across Source 1, 2, and 3, extracts 32 pairwise similarity features, and runs 5-fold GroupKFold to find the optimal decision threshold for Macro F0.5.

In [ ]:
os.makedirs("models", exist_ok=True)

# You can adjust --sample-size (e.g. 100000 or 150000 for quick balanced training, or 0 for all)
!python -m src.modeling.train \
    --s1-path data/processed/train_source1.parquet \
    --s2-path data/processed/train_source2.parquet \
    --s3-path data/processed/train_source3.parquet \
    --gt-path data/processed/train_ground_truth.parquet \
    --sample-size 150000 \
    --output-model models/lgb_model.joblib

## 4. Run Test Inference & Generate Submission TSVs
Processes all 1,732,544 test Source 1 entities across countries, applying candidate generation, feature extraction, and model thresholding.

In [ ]:
os.makedirs("output", exist_ok=True)

!python -m src.modeling.predict \
    --model-path models/lgb_model.joblib \
    --test-s1 data/processed/test_source1.parquet \
    --test-s2 data/processed/test_source2.parquet \
    --test-s3 data/processed/test_source3.parquet \
    --output-dir output

## 5. Validate Submission Files
Ensures generated TSVs adhere to the challenge rules (exact headers, 1,732,544 rows, correct prefixes).

In [ ]:
!python scripts/validate_submission.py \
    --matching output/matching_results.tsv \
    --candidate output/candidate_pairs.tsv \
    --test-s1 data/processed/test_source1.parquet

## 6. Package Submission ZIP

In [ ]:
import zipfile

zip_filename = "output/submission.zip"
with zipfile.ZipFile(zip_filename, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    z.write("output/matching_results.tsv", "matching_results.tsv")
    z.write("output/candidate_pairs.tsv", "candidate_pairs.tsv")

print(f"Submission archive successfully created at: {zip_filename}")
print(f"File size: {os.path.getsize(zip_filename) / (1024 * 1024):.2f} MB")
print("\nReady to download from JupyterLab file browser!")